In [1]:
import pandas as pd
import polars as pl
import numpy as np


In [14]:
#apreensao_drogas
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_dados = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
print(df_dados.head())

   cisp  mes   ano  mes_ano  aisp  risp           munic    mcirc   regiao  \
0     1    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
1     4    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
2     5    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
3     6    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
4     7    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   

   hom_doloso  ...  cmp  cmba  ameaca  pessoas_desaparecidas  \
0           0  ...  NaN   NaN      21                      2   
1           3  ...  NaN   NaN      15                      6   
2           3  ...  NaN   NaN      47                      2   
3           6  ...  NaN   NaN      26                      2   
4           4  ...  NaN   NaN      10                      1   

   encontro_cadaver  encontro_ossada  pol_militares_mortos_serv  \
0                 0                0                          0   
1                 

In [15]:
# Delimitar as variáveis cisp regiao munic roubo_veiculo
df_apreensao = df_dados[['cisp', 'regiao', 'munic', 'apreensao_drogas']]

# Corrigindo a Grande Niterói
correcao_acentuacao = df_apreensao['regiao'].str.startswith('Grande Niter', na=False)
# print(correcao_acentuacao)
df_apreensao.loc[correcao_acentuacao, 'regiao'] = 'Grande Niterói'
print(df_apreensao['regiao'].unique())

<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [12]:
df_apreensao = df_apreensao.groupby(['cisp', 'regiao', 'munic'], as_index=False)['apreensao_drogas'].sum()
display(df_apreensao)

,cisp,regiao,munic,apreensao_drogas
0,1,Capital,Rio de Janeiro,1303
1,4,Capital,Rio de Janeiro,5882
2,5,Capital,Rio de Janeiro,6843
3,6,Capital,Rio de Janeiro,3132
4,7,Capital,Rio de Janeiro,1519
...,...,...,...,...
142,165,Interior,Mangaratiba,1637
143,166,Interior,Angra dos Reis,7894
144,167,Capital,Paraty,74
145,167,Interior,Paraty,3350


In [25]:
array_apreensao = np.array(df_apreensao['apreensao_drogas'])

q1 = np.percentile(array_apreensao, 25)
q3 = np.percentile(array_apreensao, 75)

print('Quartil 1: ', q1)
print('Quartil 3: ', q3)

Quartil 1:  3.0
Quartil 3:  14.0


In [29]:
# Criando uma série com os maiores
df_maiores = df_apreensao[df_apreensao['apreensao_drogas'] >= q3].copy()
df_maiores['flag'] = 'mais'
display(df_maiores)

# Criando uma série com os menores
df_menores = df_apreensao[df_apreensao['apreensao_drogas'] <= q1].copy()
df_menores['flag'] = 'menos'
display(df_menores)

# Concatenando os dois dataframes df_maiores e df_menores
df_roubos_flags = pd.concat([df_maiores, df_menores], ignore_index=True)

# exportando os dados utf-8-sig p/ ler no powerbi
df_roubos_flags.to_csv('apreensoes_flags.csv', index=False, encoding='utf-8-sig')

,cisp,regiao,munic,apreensao_drogas,flag
1,4,Capital,Rio de Janeiro,35,mais
3,6,Capital,Rio de Janeiro,20,mais
8,13,Capital,Rio de Janeiro,14,mais
12,17,Capital,Rio de Janeiro,21,mais
16,21,Capital,Rio de Janeiro,23,mais
...,...,...,...,...,...
38531,146,Interior,Campos dos Goytacazes,51,mais
38533,148,Interior,Cardoso Moreira;Italva,24,mais
38534,151,Interior,Nova Friburgo,64,mais
38542,159,Interior,Cachoeiras de Macacu,18,mais


,cisp,regiao,munic,apreensao_drogas,flag
0,1,Capital,Rio de Janeiro,1,menos
4,7,Capital,Rio de Janeiro,3,menos
13,18,Capital,Rio de Janeiro,2,menos
21,26,Capital,Rio de Janeiro,0,menos
22,27,Capital,Rio de Janeiro,3,menos
...,...,...,...,...,...
38509,122,Interior,Conceição de Macabú,3,menos
38538,155,Interior,São Sebastião do Alto,1,menos
38539,156,Interior,Santa Maria Madalena,0,menos
38540,157,Interior,Trajano de Morais,0,menos
